# State-Action Drift Atlas

Questo notebook implementa una visualizzazione per diagnosticare la **distributional shift** in imitation learning quando stati e azioni sono continui e ad alta dimensionalità.

L'idea è comprimere lo spazio degli stati in una mappa 2D e sovrapporre tre segnali locali:

- **sfondo**: `log p_learner(s) / p_expert(s)`, cioè dove il learner visita più o meno stati rispetto all'expert;
- **frecce**: drift locale delle azioni, stimato come differenza tra l'azione media del learner e quella dell'expert in intorni locali della mappa;
- **punti/diagnostiche**: distanza dal manifold expert e deviazione azione-vs-expert negli stati visitati dal learner.

Nel caso SUMO, di default confronto `expert_trajectories_no_collision.pkl` con `debug_dataset_full_ep.pkl`. Se vuoi generare rollout freschi da un checkpoint SAC/PPO, imposta `LEARNER_TRAJECTORIES_PKL = None` nella configurazione.

In [ ]:
from pathlib import Path
import math
import pickle
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import TwoSlopeNorm
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "human-feedback-rl"))
sys.path.insert(0, str(REPO / "sumo-rl-ego"))

from human_feedback_rl.common.types import Transition

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"Repo: {REPO}")
print(f"UMAP disponibile: {HAS_UMAP}")

## Configurazione

La configurazione sotto è pensata per partire subito con i pickle già presenti nel repo remoto.

Per usare un checkpoint invece del pickle debug:

```python
LEARNER_TRAJECTORIES_PKL = None
CHECKPOINT_NAME = "checkpoint_0010"
```

Il notebook proverà a caricare `AGENT_ZIP` come SAC o PPO e genererà rollout nell'ambiente SUMO.

In [ ]:
# ============================ CONFIGURAZIONE ============================
SEED = 0
rng = np.random.default_rng(SEED)

DATA_DIR = REPO / "datasets"
EXPERT_TRAJECTORIES_PKL = DATA_DIR / "expert_trajectories_no_collision.pkl"

# Default veloce: dataset debug già salvato. Metti None per generare rollout da AGENT_ZIP.
LEARNER_TRAJECTORIES_PKL = DATA_DIR / "debug_dataset_full_ep.pkl"
LEARNER_LABEL = "learner/debug"

RUN_DIR = (
    REPO
    / "outputs"
    / "demo_sac_robustness_4m_maxent_corrected_no_collision_no_relabel_no_normalization"
    / "demo_sac_robustness_4m_maxent_corrected_no_collision_no_relabel_no_normalization loss=maxent_corrected relabel=False seed=0"
)
CHECKPOINT_NAME = "checkpoint_0010"
AGENT_ZIP = RUN_DIR / CHECKPOINT_NAME / "agent.zip"

ENV_ID = "HighwayEgo-v0"
ENV_KWARGS = {"ego": "continuous", "reward": "fast"}
N_ENVS = 1
N_ROLLOUT_STEPS = 20_000
AGENT_DETERMINISTIC = True

# Campionamento per l'atlante. Aumenta se vuoi figure più dense.
MAX_EXPERT_TRANSITIONS = 60_000
MAX_LEARNER_TRANSITIONS = 20_000
MAX_POINTS_PER_CLOUD = 5_000

# Proiezione: "state" mantiene la mappa come occupancy sugli stati; "state_action" include anche le azioni.
PROJECTION_INPUT = "state"  # "state" oppure "state_action"
PROJECTION_METHOD = "pca"   # "pca" oppure "umap"

# Griglia e stime locali.
GRID_BINS = 72
DENSITY_ALPHA = 0.5
MIN_BIN_TOTAL = 8
LOCAL_K = 35
QUIVER_STRIDE = 5

# Discriminatore expert-vs-learner.
CLASSIFIER_MAX_PER_CLASS = 25_000
CLASSIFIER_FEATURES = "state_action"  # "state", "action", oppure "state_action"

for path in [EXPERT_TRAJECTORIES_PKL]:
    assert path.exists(), f"Manca: {path}"
if LEARNER_TRAJECTORIES_PKL is not None:
    assert Path(LEARNER_TRAJECTORIES_PKL).exists(), f"Manca: {LEARNER_TRAJECTORIES_PKL}"
else:
    assert AGENT_ZIP.exists(), f"Manca: {AGENT_ZIP}"

print("Expert:", EXPERT_TRAJECTORIES_PKL)
print("Learner pickle:", LEARNER_TRAJECTORIES_PKL)
print("Agent zip:", AGENT_ZIP)

In [ ]:
# ============================ DATA HELPERS ============================
def is_transition_like(x):
    return hasattr(x, "observation") and hasattr(x, "action")


def load_pickle(path):
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


def flatten_with_context(source):
    """Return list[(transition, episode_id, timestep)] from trajectories or flat transitions."""
    if len(source) == 0:
        return []
    if is_transition_like(source[0]):
        trajectories = [source]
    else:
        trajectories = source

    records = []
    for episode_id, trajectory in enumerate(trajectories):
        for timestep, transition in enumerate(trajectory):
            records.append((transition, episode_id, timestep))
    return records


def sample_records(records, max_n, rng):
    records = list(records)
    if max_n is None or len(records) <= max_n:
        return records
    idx = rng.choice(len(records), size=max_n, replace=False)
    idx.sort()
    return [records[i] for i in idx]


def _stack_attr(records, attr, default=None):
    values = []
    for transition, _, _ in records:
        if hasattr(transition, attr):
            values.append(np.asarray(getattr(transition, attr), dtype=np.float32).reshape(1, -1))
        elif default is not None:
            values.append(np.asarray(default, dtype=np.float32).reshape(1, -1))
        else:
            raise AttributeError(f"Transition senza attributo {attr!r}")
    return np.vstack(values)


def records_to_arrays(records):
    obs = _stack_attr(records, "observation")
    acts = _stack_attr(records, "action")
    status = _stack_attr(records, "next_status", default=np.full(7, np.nan))
    done = np.array([float(getattr(t, "done", False)) for t, _, _ in records], dtype=np.float32)
    true_reward = np.array([float(getattr(t, "true_reward", np.nan)) for t, _, _ in records], dtype=np.float32)
    episode = np.array([ep for _, ep, _ in records], dtype=int)
    timestep = np.array([ts for _, _, ts in records], dtype=int)
    return {
        "obs": obs,
        "acts": acts,
        "status": status,
        "done": done,
        "true_reward": true_reward,
        "episode": episode,
        "timestep": timestep,
    }


def build_features(obs, acts, mode):
    if mode == "state":
        return obs, np.array([f"obs[{i}]" for i in range(obs.shape[1])])
    if mode == "action":
        return acts, np.array([f"act[{i}]" for i in range(acts.shape[1])])
    if mode == "state_action":
        names = [f"obs[{i}]" for i in range(obs.shape[1])] + [f"act[{i}]" for i in range(acts.shape[1])]
        return np.hstack([obs, acts]), np.array(names)
    raise ValueError("mode deve essere 'state', 'action' o 'state_action'")


def downsample_rows(x, max_n, rng):
    if max_n is None or len(x) <= max_n:
        return np.arange(len(x))
    return np.sort(rng.choice(len(x), size=max_n, replace=False))


def robust_z(x):
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    iqr = np.nanpercentile(x, 75) - np.nanpercentile(x, 25)
    scale = iqr if iqr > 1e-12 else np.nanstd(x)
    scale = scale if scale > 1e-12 else 1.0
    return (x - med) / scale

In [ ]:
# ============================ OPTIONAL ROLLOUT DA CHECKPOINT ============================
def collect_agent_trajectories(agent_zip, steps, seed_offset=50_000):
    """Generate learner trajectories from an SB3 SAC/PPO checkpoint.

    Questa funzione viene chiamata solo se LEARNER_TRAJECTORIES_PKL = None.
    """
    import torch as th
    from stable_baselines3 import PPO, SAC
    from human_feedback_rl.common.reward_nets import RewardNet
    from human_feedback_rl.common.trajectory_generators import TrajectoryGeneratorFromAgent
    import sumo_rl_ego as sre

    try:
        import sumo_gym_ego.core.simulation as _sim_mod
        import traci as _traci_mod
        _sim_mod.load_traci = lambda use_gui: _traci_mod
    except Exception as exc:
        print("Patch traci non applicata:", repr(exc))

    class ZeroRewardModel(RewardNet):
        def forward(self, state, action, next_status=None, done=None):
            return th.zeros(state.shape[0], dtype=th.float32, device=state.device)

    env = sre.make_vec_env(
        ENV_ID,
        n_envs=N_ENVS,
        base_seed=SEED + seed_offset,
        **ENV_KWARGS,
    )
    try:
        last_error = None
        agent = None
        for algo in (SAC, PPO):
            try:
                agent = algo.load(agent_zip, env=env, device="cpu")
                print(f"Caricato {agent_zip} come {algo.__name__}")
                break
            except Exception as exc:
                last_error = exc
        if agent is None:
            raise RuntimeError(f"Non riesco a caricare {agent_zip}") from last_error

        reward_model = ZeroRewardModel(env.observation_space, env.action_space)
        generator = TrajectoryGeneratorFromAgent(agent=agent, reward_model=reward_model, venv=env)
        return generator.sample(steps)
    finally:
        env.close()

In [ ]:
# ============================ CARICAMENTO DATI ============================
expert_source = load_pickle(EXPERT_TRAJECTORIES_PKL)
if LEARNER_TRAJECTORIES_PKL is None:
    learner_source = collect_agent_trajectories(AGENT_ZIP, N_ROLLOUT_STEPS)
    LEARNER_LABEL = CHECKPOINT_NAME
else:
    learner_source = load_pickle(LEARNER_TRAJECTORIES_PKL)

expert_records_all = flatten_with_context(expert_source)
learner_records_all = flatten_with_context(learner_source)

expert_records = sample_records(expert_records_all, MAX_EXPERT_TRANSITIONS, rng)
learner_records = sample_records(learner_records_all, MAX_LEARNER_TRANSITIONS, rng)

E = records_to_arrays(expert_records)
L = records_to_arrays(learner_records)

print(f"Expert:  {len(expert_records):,}/{len(expert_records_all):,} transizioni campionate")
print(f"Learner: {len(learner_records):,}/{len(learner_records_all):,} transizioni campionate")
print("obs_dim:", E["obs"].shape[1], "act_dim:", E["acts"].shape[1])
print("Expert episodes:", len(np.unique(E["episode"])), "Learner episodes:", len(np.unique(L["episode"])))

## Proiezione comune

La mappa viene fittata una sola volta su expert + learner, così i due insiemi vivono nello stesso sistema di coordinate.

Default: la proiezione usa solo gli stati (`PROJECTION_INPUT = "state"`). Le azioni entrano poi come campo vettoriale locale.

In [ ]:
# ============================ PROIEZIONE 2D E DIAGNOSTICHE PER-PUNTO ============================
expert_projection_features, projection_feature_names = build_features(E["obs"], E["acts"], PROJECTION_INPUT)
learner_projection_features, _ = build_features(L["obs"], L["acts"], PROJECTION_INPUT)

projection_scaler = StandardScaler()
all_projection_features = np.vstack([expert_projection_features, learner_projection_features])
all_scaled = projection_scaler.fit_transform(all_projection_features)

if PROJECTION_METHOD == "umap":
    if not HAS_UMAP:
        raise ImportError("umap-learn non è installato. Usa PROJECTION_METHOD='pca' oppure installa umap-learn.")
    reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=35, min_dist=0.10)
elif PROJECTION_METHOD == "pca":
    reducer = PCA(n_components=2, random_state=SEED)
else:
    raise ValueError("PROJECTION_METHOD deve essere 'pca' o 'umap'.")

xy_all = reducer.fit_transform(all_scaled)
expert_xy = xy_all[: len(E["obs"])]
learner_xy = xy_all[len(E["obs"]):]

if PROJECTION_METHOD == "pca":
    print("Explained variance ratio:", reducer.explained_variance_ratio_)

# Azioni standardizzate: il drift locale e la distanza azione usano una scala comune.
action_scaler = StandardScaler()
all_actions_scaled = action_scaler.fit_transform(np.vstack([E["acts"], L["acts"]]))
expert_action_scaled = all_actions_scaled[: len(E["acts"])]
learner_action_scaled = all_actions_scaled[len(E["acts"]):]

n_action_components = min(2, E["acts"].shape[1])
action_pca = PCA(n_components=n_action_components, random_state=SEED).fit(all_actions_scaled)
print("Action PCA explained variance:", action_pca.explained_variance_ratio_)

# Distanza OOD: quanto ogni stato learner è lontano dai vicini expert nello spazio stato standardizzato.
state_scaler = StandardScaler().fit(E["obs"])
expert_state_scaled = state_scaler.transform(E["obs"])
learner_state_scaled = state_scaler.transform(L["obs"])
state_k = min(LOCAL_K, len(expert_state_scaled))
state_nn = NearestNeighbors(n_neighbors=state_k).fit(expert_state_scaled)
learner_state_distances, learner_state_neighbors = state_nn.kneighbors(learner_state_scaled)
learner_expert_state_distance = learner_state_distances.mean(axis=1)

# Deviazione azione: azione learner vs azione media expert negli stati vicini sulla mappa 2D.
xy_k = min(LOCAL_K, len(expert_xy))
xy_nn_expert = NearestNeighbors(n_neighbors=xy_k).fit(expert_xy)
_, learner_local_expert_idx = xy_nn_expert.kneighbors(learner_xy)
local_expert_action_mean = expert_action_scaled[learner_local_expert_idx].mean(axis=1)
learner_action_deviation = np.linalg.norm(learner_action_scaled - local_expert_action_mean, axis=1)

print("Projection shape expert/learner:", expert_xy.shape, learner_xy.shape)

In [ ]:
# ============================ GRIGLIA: DENSITY SHIFT + ACTION DRIFT FIELD ============================
def make_grid(expert_xy, learner_xy, bins=72, pad_frac=0.04):
    xy = np.vstack([expert_xy, learner_xy])
    xmin, ymin = xy.min(axis=0)
    xmax, ymax = xy.max(axis=0)
    xpad = (xmax - xmin) * pad_frac + 1e-8
    ypad = (ymax - ymin) * pad_frac + 1e-8
    xedges = np.linspace(xmin - xpad, xmax + xpad, bins + 1)
    yedges = np.linspace(ymin - ypad, ymax + ypad, bins + 1)
    return xedges, yedges


def density_shift_grid(expert_xy, learner_xy, xedges, yedges, alpha=0.5, min_total=8):
    H_e, _, _ = np.histogram2d(expert_xy[:, 0], expert_xy[:, 1], bins=[xedges, yedges])
    H_l, _, _ = np.histogram2d(learner_xy[:, 0], learner_xy[:, 1], bins=[xedges, yedges])
    n_cells = H_e.size
    P_e = (H_e + alpha) / (H_e.sum() + alpha * n_cells)
    P_l = (H_l + alpha) / (H_l.sum() + alpha * n_cells)
    log_ratio = np.log(P_l / P_e)
    support = (H_e + H_l) >= min_total
    return log_ratio, support, H_e, H_l


def grid_values_at_points(xy, xedges, yedges, values):
    ix = np.searchsorted(xedges, xy[:, 0], side="right") - 1
    iy = np.searchsorted(yedges, xy[:, 1], side="right") - 1
    valid = (ix >= 0) & (ix < values.shape[0]) & (iy >= 0) & (iy < values.shape[1])
    out = np.full(len(xy), np.nan, dtype=float)
    out[valid] = values[ix[valid], iy[valid]]
    return out


def action_drift_field(
    expert_xy,
    learner_xy,
    expert_action_scaled,
    learner_action_scaled,
    action_pca,
    xedges,
    yedges,
    support,
    stride=5,
    k=35,
):
    centers_x = 0.5 * (xedges[:-1] + xedges[1:])
    centers_y = 0.5 * (yedges[:-1] + yedges[1:])
    nn_e = NearestNeighbors(n_neighbors=min(k, len(expert_xy))).fit(expert_xy)
    nn_l = NearestNeighbors(n_neighbors=min(k, len(learner_xy))).fit(learner_xy)

    xs, ys, us, vs, mags = [], [], [], [], []
    components = action_pca.components_
    for ix in range(0, len(centers_x), stride):
        for iy in range(0, len(centers_y), stride):
            if not support[ix, iy]:
                continue
            center = np.array([[centers_x[ix], centers_y[iy]]])
            _, idx_e = nn_e.kneighbors(center)
            _, idx_l = nn_l.kneighbors(center)
            delta = learner_action_scaled[idx_l[0]].mean(axis=0) - expert_action_scaled[idx_e[0]].mean(axis=0)
            projected_delta = delta @ components.T
            u = float(projected_delta[0])
            v = float(projected_delta[1]) if len(projected_delta) > 1 else 0.0
            xs.append(centers_x[ix])
            ys.append(centers_y[iy])
            us.append(u)
            vs.append(v)
            mags.append(float(np.linalg.norm(delta)))

    return map(np.asarray, (xs, ys, us, vs, mags))


xedges, yedges = make_grid(expert_xy, learner_xy, bins=GRID_BINS)
log_ratio_grid, support_grid, expert_counts, learner_counts = density_shift_grid(
    expert_xy, learner_xy, xedges, yedges, alpha=DENSITY_ALPHA, min_total=MIN_BIN_TOTAL
)
learner_log_density_shift = grid_values_at_points(learner_xy, xedges, yedges, log_ratio_grid)

field_x, field_y, field_u, field_v, field_mag = action_drift_field(
    expert_xy,
    learner_xy,
    expert_action_scaled,
    learner_action_scaled,
    action_pca,
    xedges,
    yedges,
    support_grid,
    stride=QUIVER_STRIDE,
    k=LOCAL_K,
)

print("Celle supportate:", int(support_grid.sum()), "/", support_grid.size)
print("Frecce drift:", len(field_x))

## Atlante Stato-Azione della Deriva

Come leggere la figura:

- rosso: il learner visita quella regione più dell'expert;
- blu: regione prevalentemente expert;
- bianco: occupancy simile;
- frecce: drift locale dell'azione media. La direzione è nello spazio PCA delle azioni, ancorata alla posizione dello stato sulla mappa;
- intensità delle frecce: norma del drift azione nello spazio azione standardizzato.

In [ ]:
# ============================ FIGURA 1: STATE-ACTION DRIFT ATLAS ============================
def draw_density_contours(ax, xy, color, label, bins=70, levels=6, alpha=0.8, linewidth=1.1):
    counts, xe, ye = np.histogram2d(xy[:, 0], xy[:, 1], bins=bins)
    positive = counts[counts > 0]
    if len(positive) == 0:
        return
    contour_levels = np.unique(np.quantile(positive, np.linspace(0.50, 0.96, levels)))
    if len(contour_levels) == 0:
        return
    xc = 0.5 * (xe[:-1] + xe[1:])
    yc = 0.5 * (ye[:-1] + ye[1:])
    ax.contour(xc, yc, counts.T, levels=contour_levels, colors=color, alpha=alpha, linewidths=linewidth)
    ax.plot([], [], color=color, lw=linewidth, label=label)


fig, ax = plt.subplots(figsize=(11, 8.5))

masked_log_ratio = np.where(support_grid, log_ratio_grid, np.nan)
finite_vals = np.abs(masked_log_ratio[np.isfinite(masked_log_ratio)])
limit = np.nanpercentile(finite_vals, 97) if len(finite_vals) else 1.0
limit = max(limit, 1e-6)
norm = TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)

mesh = ax.pcolormesh(
    xedges,
    yedges,
    masked_log_ratio.T,
    shading="auto",
    cmap="coolwarm",
    norm=norm,
    alpha=0.78,
)
cb = fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.02)
cb.set_label("log density ratio: learner / expert")

idx_e_plot = downsample_rows(expert_xy, MAX_POINTS_PER_CLOUD, rng)
idx_l_plot = downsample_rows(learner_xy, MAX_POINTS_PER_CLOUD, rng)
ax.scatter(expert_xy[idx_e_plot, 0], expert_xy[idx_e_plot, 1], s=6, c="#444444", alpha=0.12, linewidths=0, label="expert points")
ax.scatter(learner_xy[idx_l_plot, 0], learner_xy[idx_l_plot, 1], s=7, c="#111111", alpha=0.18, linewidths=0, label=f"{LEARNER_LABEL} points")

draw_density_contours(ax, expert_xy, color="#333333", label="expert density", alpha=0.55)
draw_density_contours(ax, learner_xy, color="#ffb000", label=f"{LEARNER_LABEL} density", alpha=0.85)

if len(field_x):
    raw_norm = np.sqrt(field_u**2 + field_v**2)
    span = max(xedges[-1] - xedges[0], yedges[-1] - yedges[0])
    scale_factor = 0.045 * span / (np.nanpercentile(raw_norm, 90) + 1e-8)
    quiv = ax.quiver(
        field_x,
        field_y,
        field_u * scale_factor,
        field_v * scale_factor,
        field_mag,
        cmap="viridis",
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.004,
        alpha=0.95,
    )
    qcb = fig.colorbar(quiv, ax=ax, fraction=0.046, pad=0.08)
    qcb.set_label("local action drift norm")

ax.set_title(f"State-Action Drift Atlas ({PROJECTION_METHOD.upper()}, projection={PROJECTION_INPUT})")
ax.set_xlabel("atlas dimension 1")
ax.set_ylabel("atlas dimension 2")
ax.legend(frameon=False, loc="best")
plt.tight_layout()
plt.show()

## Test discriminativo expert-vs-learner

Questo è un controllo quantitativo: addestro un classificatore a distinguere transizioni expert da transizioni learner.

Interpretazione rapida:

- `AUC ≈ 0.5`: distribuzioni difficili da distinguere;
- `AUC 0.7-0.8`: shift moderata;
- `AUC > 0.9`: shift evidente.

In [ ]:
# ============================ CLASSIFIER TWO-SAMPLE TEST ============================
def sample_equal_per_class(X_e, X_l, max_per_class, rng):
    n = min(len(X_e), len(X_l), max_per_class)
    ie = rng.choice(len(X_e), size=n, replace=False)
    il = rng.choice(len(X_l), size=n, replace=False)
    X = np.vstack([X_e[ie], X_l[il]])
    y = np.r_[np.zeros(n, dtype=int), np.ones(n, dtype=int)]
    return X, y


X_e_cls, cls_feature_names = build_features(E["obs"], E["acts"], CLASSIFIER_FEATURES)
X_l_cls, _ = build_features(L["obs"], L["acts"], CLASSIFIER_FEATURES)
X_cls, y_cls = sample_equal_per_class(X_e_cls, X_l_cls, CLASSIFIER_MAX_PER_CLASS, rng)

X_train, X_test, y_train, y_test = train_test_split(
    X_cls,
    y_cls,
    test_size=0.30,
    random_state=SEED,
    stratify=y_cls,
)
cls_scaler = StandardScaler().fit(X_train)
clf = LogisticRegression(max_iter=2_000, class_weight="balanced", solver="lbfgs")
clf.fit(cls_scaler.transform(X_train), y_train)
proba = clf.predict_proba(cls_scaler.transform(X_test))[:, 1]
pred = (proba >= 0.5).astype(int)
auc = roc_auc_score(y_test, proba)
acc = accuracy_score(y_test, pred)

coef = np.abs(clf.coef_[0])
importance = pd.DataFrame({"feature": cls_feature_names, "abs_coef": coef}).sort_values("abs_coef", ascending=False)

print(f"Classifier features: {CLASSIFIER_FEATURES}")
print(f"AUC={auc:.3f}  accuracy={acc:.3f}  n_test={len(y_test):,}")
display(importance.head(15))

## Quando avviene la shift nel rollout?

Qui riassumo, per timestep del learner, tre segnali:

- `expert_state_distance`: distanza media dai vicini expert nello spazio stato standardizzato;
- `action_deviation`: differenza tra azione learner e azione media expert locale;
- `log_density_shift`: occupancy ratio locale sulla mappa.

Se una curva cresce nel tempo, è un segnale tipico di errore che si accumula in closed-loop.

In [ ]:
# ============================ FIGURA 2: SHIFT NEL TEMPO ============================
time_df = pd.DataFrame({
    "episode": L["episode"],
    "timestep": L["timestep"],
    "expert_state_distance": learner_expert_state_distance,
    "action_deviation": learner_action_deviation,
    "log_density_shift": learner_log_density_shift,
    "true_reward": L["true_reward"],
})

n_time_bins = min(40, max(5, int(time_df["timestep"].nunique())))
time_df["time_bin"] = pd.cut(time_df["timestep"], bins=n_time_bins, duplicates="drop")
time_summary = (
    time_df
    .groupby("time_bin", observed=True)
    .agg(
        timestep=("timestep", "mean"),
        expert_state_distance=("expert_state_distance", "mean"),
        action_deviation=("action_deviation", "mean"),
        log_density_shift=("log_density_shift", "mean"),
        n=("timestep", "size"),
    )
    .reset_index(drop=True)
)

display(time_summary.head())

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
metrics = [
    ("expert_state_distance", "Distance to expert state manifold"),
    ("action_deviation", "Local action deviation"),
    ("log_density_shift", "Local log density ratio"),
]
for ax, (col, label) in zip(axes, metrics):
    ax.plot(time_summary["timestep"], time_summary[col], lw=2)
    ax.scatter(time_summary["timestep"], time_summary[col], s=18)
    ax.set_ylabel(label)
    ax.grid(alpha=0.18)
axes[-1].set_xlabel("learner timestep")
fig.suptitle("Distributional shift lungo il rollout learner", y=1.02)
plt.tight_layout()
plt.show()

## Episodi e punti più rischiosi

Il punteggio `risk_score` combina distanza OOD, deviazione d'azione e densità learner-dominante. Serve per trovare esempi da ispezionare o da riportare in tesi.

In [ ]:
# ============================ TABLE: TOP SHIFT POINTS ============================
positive_log_shift = np.nan_to_num(np.maximum(learner_log_density_shift, 0.0), nan=0.0)
risk_score = (
    robust_z(learner_expert_state_distance)
    + robust_z(learner_action_deviation)
    + robust_z(positive_log_shift)
)

status_id = np.nanargmax(L["status"], axis=1) if L["status"].ndim == 2 else np.full(len(risk_score), -1)
summary = pd.DataFrame({
    "episode": L["episode"],
    "timestep": L["timestep"],
    "risk_score": risk_score,
    "expert_state_distance": learner_expert_state_distance,
    "action_deviation": learner_action_deviation,
    "log_density_shift": learner_log_density_shift,
    "status_id": status_id,
    "true_reward": L["true_reward"],
})
for j in range(L["acts"].shape[1]):
    summary[f"act[{j}]"] = L["acts"][:, j]
for j in range(min(6, L["obs"].shape[1])):
    summary[f"obs[{j}]"] = L["obs"][:, j]

top_shift = summary.sort_values("risk_score", ascending=False).head(25)
display(top_shift)

## Tracce episodiche sulla mappa

Questa figura mostra alcuni episodi learner direttamente sull'atlante. È utile per capire se la policy parte dentro il manifold expert e poi scivola verso regioni learner-dominanti.

In [ ]:
# ============================ FIGURA 3: TRAIETTORIE LEARNER SULL'ATLANTE ============================
N_EPISODES_TO_TRACE = 8
learner_episodes = np.unique(L["episode"])
selected_episodes = learner_episodes[:N_EPISODES_TO_TRACE]

fig, ax = plt.subplots(figsize=(10, 8))
mesh = ax.pcolormesh(
    xedges,
    yedges,
    masked_log_ratio.T,
    shading="auto",
    cmap="coolwarm",
    norm=norm,
    alpha=0.62,
)
fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.02, label="log density ratio")
draw_density_contours(ax, expert_xy, color="#222222", label="expert density", alpha=0.65)

colors = plt.cm.tab10(np.linspace(0, 1, len(selected_episodes)))
for episode_id, color in zip(selected_episodes, colors):
    mask = L["episode"] == episode_id
    order = np.argsort(L["timestep"][mask])
    xy = learner_xy[mask][order]
    if len(xy) == 0:
        continue
    ax.plot(xy[:, 0], xy[:, 1], color=color, lw=1.8, alpha=0.85, label=f"ep {episode_id}")
    ax.scatter(xy[0, 0], xy[0, 1], color=color, s=55, marker="o", edgecolor="white", linewidth=0.8)
    ax.scatter(xy[-1, 0], xy[-1, 1], color=color, s=70, marker="X", edgecolor="white", linewidth=0.8)

ax.set_title(f"Tracce learner sull'atlante: start=o, end=X ({LEARNER_LABEL})")
ax.set_xlabel("atlas dimension 1")
ax.set_ylabel("atlas dimension 2")
ax.legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## Variante: confronto su feature interpretabili

La mappa astratta è utile per vedere il manifold. Questa cella serve invece per fare una figura più leggibile in tesi, scegliendo due feature originali.

In [ ]:
# ============================ FIGURA 4: FEATURE ORIGINALI SCELTE A MANO ============================
X_FEATURE = "obs[0]"
Y_FEATURE = "act[0]"

all_original_features_e, original_feature_names = build_features(E["obs"], E["acts"], "state_action")
all_original_features_l, _ = build_features(L["obs"], L["acts"], "state_action")

def feature_index(name, names):
    matches = np.where(names == name)[0]
    if len(matches) == 0:
        raise ValueError(f"Feature {name!r} non trovata. Disponibili: {names.tolist()}")
    return int(matches[0])

xi = feature_index(X_FEATURE, original_feature_names)
yi = feature_index(Y_FEATURE, original_feature_names)
idx_e = downsample_rows(all_original_features_e, MAX_POINTS_PER_CLOUD, rng)
idx_l = downsample_rows(all_original_features_l, MAX_POINTS_PER_CLOUD, rng)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(all_original_features_e[idx_e, xi], all_original_features_e[idx_e, yi], s=8, c="#444444", alpha=0.18, linewidths=0, label="expert")
sc = ax.scatter(
    all_original_features_l[idx_l, xi],
    all_original_features_l[idx_l, yi],
    s=10,
    c=risk_score[idx_l],
    cmap="magma",
    alpha=0.55,
    linewidths=0,
    label=LEARNER_LABEL,
)
fig.colorbar(sc, ax=ax, label="risk score")
ax.set_xlabel(X_FEATURE)
ax.set_ylabel(Y_FEATURE)
ax.set_title(f"Feature view: {X_FEATURE} vs {Y_FEATURE}")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Nota metodologica

Questa visualizzazione non pretende di stimare densità ad alta dimensionalità in modo assoluto. Usa invece una lettura locale e comparativa:

```text
rho_E(s, a) vs rho_pi(s, a)
```

- la proiezione rende leggibile il supporto degli stati;
- il log-ratio evidenzia dove cambia l'occupancy;
- il campo vettoriale separa la shift di stato dalla shift di azione;
- il classificatore dà un numero unico per verificare che la separazione non sia solo un artefatto visuale.

Per una figura finale da tesi, salva l'atlante con `fig.savefig(...)` nella cella corrispondente dopo aver scelto il run/checkpoint definitivo.